In [ ]:
#from google.colab import drive # remove the cell if not using colab
#drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
base_path = Path('.')

# Klasyfikacja pasażerów Titanica
Nie wiemy czy dla DiCaprio było miejsce na drzwiach, ale wiemy że gdyby był tam Wojfer87 to by z nimi wyciskał pompki na górze lodowej. Teraz twoja pora na wyciskanie.
# Twoje zadanie to:
**stworzenie modelu przewidującego szanse przeżycia katastrofy Titanica**.

![https://i1.jbzd.com.pl/contents/2025/11/normal/v5Fth4DcPpPPxSrXQ5rCbAgZ8EifWiiF.png](https://i1.jbzd.com.pl/contents/2025/11/normal/v5Fth4DcPpPPxSrXQ5rCbAgZ8EifWiiF.png "Wojfer")



#### Twoim celem będzie jest wytrenowanie modeli do klasyfikacji każdego pasażera Titanica jako ofiary (0) lub osoby, która przeżyła (1).

Poniżej znajdziesz pytania, które mogą być pomocne w zadaniu:

- Czego nauczyło Cię o badanym zbiorze danych poprzednie zadanie? Jak możesz wykorzystać wyciągnięte z niego wnioski w procesie tworzenia modelu?
- Jak przeprowadzenie standaryzacji danych może wpływać na zachowanie modelu?
- Co mój model robi i w jaki sposób?
- Jak nie przetrenować wybranego modelu?
- Jaki wynik klasyfikacji możemy uznać za *dobry*?


Wymagania:
- Wypisz obserwacje z pierwszego zadania, które pomogą Ci w tym. Co było przydatne, a co okazało się bezużyteczne?
- [Nie doprowadź](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) do ~~przecieku statku~~ wycieku danych (np. nie ucz modelu na danych testowych). Nauczone modele odpal na danych treningowych i testowych - opisz uzyskane wyniki.
- Stwórz baseline, czyli dla porównania sprawdź jak z zadaniem radzi sobie [Dummy Classifier](https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyClassifier.html) (jeśli Twój docelowy model radzi sobie gorzej - uciekaj)
- Przeprowadź badania na dwóch wybranych modelach uczenia maszynowego (np. spośród: drzew decyzyjnych, SVM, MLP, KNN, z gwiazdką [XGBoost](https://xgboost.readthedocs.io/en/stable))
- W badaniach użyj wybranych metryk klasyfikacji. Wybór uzasadnij.
- Dla każdego modelu wybierz co najmniej dwa hiperparametry i przeprowadź badania zależności wyników metryk klasyfikacji od wartości hiperparametrów. Zwizualizuj wszystko ładnie, zastanów się dlaczego tak mogło być i wyciągnij i wypisz wnioski.
- Podsumuj przeprowadzone badania, wypisz wnioski.

Niezmiennie, zadbaj o czytelność kodu i nazewnictwo zmiennych. Jeśli jakiś wycinek kodu się powtarza, to wyodrębnij go do funkcji. Postaraj się zamieszczać swoje wnioski w postaci komentarza `Markdown`.

Jeśli chcesz, możesz sprawdzić (przyjmując pewne założenia), jakie byłyby Twoje szanse na Titanicu.

Uwaga! Jeśli Titanic to dla Ciebie nic i baaaaardzo chcesz to możesz w ramach tego zadania zająć się [bardziej wymagającym](https://archive.ics.uci.edu/dataset/365/polish+companies+bankruptcy+data) zbiorem.

In [ ]:
titanic_df = pd.read_csv("./titanic_preprocessed.csv")

In [ ]:
titanic_df.head()

In [ ]:
titanic_df.columns

### Podsumowanie zadania 1, wybór metryk klasyfikacji
Cechy kategorialne "Name", "Ticket", "Cabin" same w sobie nie dostarczają dodatkowych informacji. Po wstępnym przetworzeniu:
- cecha Ticket pomaga uzupełnienie brakujących wartości w cesze Fare, dodatkowo wywodzi się z niej cecha numeryczna dyskretna TicketGroupSize
- cecha HasCabin wywodząca się od Cabin determinuje, czy dany pasażer miał przypisaną kajutę

Najprawdpodobniej zbiór danych nie posiada szumów informacyjnych. Wartości odstające w cechach numerycznych są jak najbardziej prawdopodobne ze względu na fakt, że zbiór danych opiera się na rzeczywistej liście pasażerów statku Titanic z początku XX wieku. Wszelkie skrajne obserwacje np. bardzo wysokie ceny biletów Fare w pierwszej klasie lub szeroki przekrój wiekowy Age odzwierciedlają realia historyczne. Jednakże, w celu uniknięcia utrudnień związanych z uczeniem modeli na tym zbiorze danych obserwacje z najbardziej odstającymi wartościami w cechach numerycznych zostały pominięte.

Większość cech numerycznych charakteryzuje się rozkładem skośnym prawostronnie.

Jako metrykę klasyfikacji wytrenowanych modeli przyjmuję precyzję (ang. precision) dla kategorii ocalałych (1) daną wzorem:
$$\text{Precision} = \frac{\text{TP}}{\text{TP} + \text{FP}}$$

W przypadku zbioru danych pasażerów Titanica zależy nam na tym, aby nie budzić fałszywych nadziei wśród rodzin pasażerów. Precyzja uwzględnia błędy typu False Positive (przypadki, w których model niesłusznie sklasyfikował ofiarę jako ocalałego). Im wyższa precyzja, tym mniej fałszywych alarmów, co oznacza, że model rzadko przypisuje status ocalałego komuś, kto w rzeczywistości zginął, zapewniając wysoką wiarygodność pozytywnych predykcji. 

Poniższa metryka:
$$\text{F1}= 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$
będzie weryfikować wiarygodność precyzji. Jeśli precyzja jest wysoka, ale czułość drastycznie spada, bo model przypisuje większość ocalałym kategorię 0 F1 obniży się.

### Podział zbioru danych na zbiór treningowy i zbiór testowy

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(titanic_df.drop(columns=['Survived']), titanic_df['Survived'], test_size=0.2, random_state=42, stratify=titanic_df['Survived'])

### Transformacja i skalowanie cech numerycznych

Decyzja co do transformacji i skalowania cech zostały podjęte na podstawie wykresów oraz ich wniosków z poprzedniego zadania.

In [ ]:
from sklearn.preprocessing import StandardScaler, RobustScaler

X_train['Fare'] = np.log1p(X_train['Fare'])
X_test['Fare'] = np.log1p(X_test['Fare'])

to_robust = ['Age', 'SibSp', 'Parch', 'FamilySize', 'TicketGroupSize']

rob_scaler = RobustScaler()

X_train[to_robust] = rob_scaler.fit_transform(X_train[to_robust])
X_test[to_robust] = rob_scaler.transform(X_test[to_robust])

std_scaler = StandardScaler()

X_train[['Fare']] = std_scaler.fit_transform(X_train[['Fare']])
X_test[['Fare']] = std_scaler.transform(X_test[['Fare']])


### DummyClassifier

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report

In [ ]:
strategies = ["most_frequent", "stratified", "uniform"]

for strategy in strategies:
    dummy_model = DummyClassifier(strategy=strategy)
    dummy_model.fit(X_train, y_train)
    y_pred = dummy_model.predict(X_test)
    
    print(strategy.capitalize())
    print(classification_report(y_test, y_pred))
    print("-------------------")

### Pierwszy trening modeli
Spośród modeli wybrałem SVC, KNN i DecisionTree. Poniżej przykładowe wyniki modeli z losowo wybranymi hiperparametrami. 

In [ ]:
from sklearn.svm import SVC

svc_model = SVC(kernel = 'linear', random_state=42)
svc_model.fit(X_train, y_train)
y_pred = svc_model.predict(X_test)

print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=2, metric='cosine')
knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(max_depth=2, random_state=42)
dt_model.fit(X_train, y_train)

y_pred = dt_model.predict(X_test)
print(classification_report(y_test, y_pred))

SVC i KNN uzyskały dokładnie takie same wyniki precyzji (0,81) i F1 (0,73) na zbiorze testowym. Zapewniają one lepszy ogólny balans i rzadziej pomijają faktycznych ocalałych w porównaniu do drzewa decyzyjnego. Drzewo decyzyjne ma najwyższą precyzję (0,83), ale kosztem czułości. SVC i KNN wydają się pozornie lepsze, natomiast do wszystkich trzech modeli hiperparametry albo zostały wylosowane metodą nieempiryczną, albo zostały ustawione domyślne wartości.

### Dobór hiperparametrów za pomocą GridSearchCV

In [ ]:
def print_report(model_find):
    print(model_find.best_params_)
    print(f"Najlepszy wynik: {round(model_find.best_score_,3)}")
    print(f"Wynik na zbiorze testowym: {model_find.score(X_test, y_test)}")

In [ ]:
def prepared_tabel(model_find, first_p, second_p, metric):
    tabel = pd.DataFrame(model_find.cv_results_)
    tabel = tabel[[f"param_{first_p}", f"param_{second_p}", f'mean_test_{metric}', 'mean_test_f1', f'rank_test_{metric}']]
    return tabel.sort_values(f'mean_test_{metric}', ascending=False).head(10)

In [ ]:
def show_plot(df, parameter, metric):
    plt.figure(figsize=(10, 6))
    sns.lineplot(data=df, x=f'param_{parameter}', y=f'mean_test_{metric}', errorbar=None)
    plt.title(f'Wpływ {parameter} na {metric} modelu')
    plt.show() 

In [ ]:
def show_heatmap(df, first_p, second_p, metric):
    pivot_table = df.pivot(
        index=f'param_{first_p}',
        columns=f'param_{second_p}',
        values=f'mean_test_{metric}'
    )

    plt.figure(figsize=(10,5))
    sns.heatmap(pivot_table, annot=True, cmap='coolwarm', cbar= True, fmt='.3f')
    plt.title(f'Wpływ obu hiperparametrów na {metric} modelu')
    plt.xlabel(f'{second_p}')
    plt.ylabel(f'{first_p}')
    plt.show()

In [ ]:
from sklearn.model_selection import GridSearchCV

svc_grid = {
    'kernel': ['linear', 'poly', 'rbf', 'sigmoid'],
    'tol': np.arange(0.01,0.9,0.01)
}

svc_find = GridSearchCV(
    SVC(random_state=42),
    svc_grid,
    cv=3,
    scoring = ['precision', 'f1'],
    refit = 'precision',
    n_jobs=2
)

svc_find.fit(X_train, y_train)

print_report(svc_find)
print(prepared_tabel(svc_find, "kernel", "tol", "precision"))

results_df = pd.DataFrame(svc_find.cv_results_)

for hiper in ["kernel", "tol"]:
    show_plot(results_df, hiper, "precision")


Dla ('linear', 0.30) osiągnięto:
- precision = 0.758905, niższa niż w pierwszym treningu (0,81)
- f1 = 0.703918, porównywalna do pierwszego treningu (0,73)


In [ ]:
from sklearn.model_selection import GridSearchCV

knn_grid = {
    'n_neighbors': range(3,9),
    'metric': ['euclidean', 'l2', 'manhattan', 'l1', 'cityblock', 'chebyshev', 'infinity', 'minkowski']
}

knn_find = GridSearchCV(
    KNeighborsClassifier(),
    knn_grid,
    cv=3,
    scoring = ['precision', 'f1'],
    refit = 'precision',
    n_jobs=2
)

knn_find.fit(X_train, y_train)

print_report(knn_find)

print(prepared_tabel(knn_find, "n_neighbors", "metric", "precision"))

results_df = pd.DataFrame(knn_find.cv_results_)

for hiper in ["n_neighbors", "metric"]:
    show_plot(results_df, hiper, "precision")

show_heatmap(results_df, "n_neighbors", "metric", "precision")

Dla ('manhattan', 4) osiągnięto:
- precision = 0.783052 vs 0.81
- f1 = 0.636616 vs 0.73

In [ ]:
from sklearn.model_selection import GridSearchCV

dt_grid = {
    'criterion': ['gini', 'entropy', 'log_loss'],
    'max_depth': range(2,12)
}

dt_find = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    dt_grid,
    cv=3,
    scoring = ['precision', 'f1'],
    refit = 'precision',
    n_jobs=2
)

dt_find.fit(X_train, y_train)

print_report(dt_find)

print(prepared_tabel(dt_find, "criterion", "max_depth", "precision"))

results_df = pd.DataFrame(dt_find.cv_results_)

for hiper in ["criterion", "max_depth"]:
    show_plot(results_df, hiper, "precision")

show_heatmap(results_df, 'criterion', 'max_depth', 'precision')

Dla ('gini', 2) osiągnięto:
- precision:  0.944361   
- f1: 0.591796

### Wnioski końcowe

Wartości hiperparametrów poszczególnych modeli dawały zawsze dawały uśrednioną precyzję i f1 niższą od wartości domyślnych z pierwszego treningu. Jest tak, ponieważ bazowy trening opierał się na 80% danych, natomiast treningi służące do poszukiwania optymalnych wartości hiperparametrów były przeprowadzane iteracyjnie poprzez walidację krzyżową. Spośród trzech treningów musiał zaistnieć przynajmniej taki jeden, w którym zbiór danych był niewłaściwie podzielony, co zaniżało ostatecznie uśrednioną precyzję i f1 wytrenowanych modeli.

Wykresy lineplot obrazują nauczenie się modelu w zależności od wartości hiperparametru. Należy mieć na uwadze, że poszukiwaliśmy optymalnych wartości dla dwóch hiperparametrów jednocześnie, w związku z czym do zobrazowania zależności wymagana jest heatmapa. Domyślnie każda wartość y wykresu jest uśrednioną wartością wszystkich możliwości drugiego parametru.

Wartości hiperparametrów znalezione przez GridSearchCV dla SVC i KNN podwyższyły zarówno precyzję i f1 o kilka setnych. Niestety hiperparametry znalezione dla DT o ile podwyższyły precyzję do 0,96 to niestety f1 spadł z 0,65 do 0,58. Przy takich parametrach model ten rzadko decydował się na wskazanie ocalonych pasażerów jako ocalonych, przez co generował niewiele fałszywych alarmów (False Positives). Wiązało się to jednak z jednoczesnym pominięciem dużej liczby rzeczywistych przypadków pozytywnych, co obniżyło czułość (recall), a co za tym idzie – miarę F1. Taka dysproporcja sprawiła, że pomimo bardzo dobrej precyzji, ogólna skuteczność modelu ujęta w mierze F1 uległa pogorszeniu, co pokazuje, jak niewystarczająca bywa sama metryka precyzji bez uwzględnienia czułości czy miary F

